# Baseline CVAE Test

In [ ]:
from pathlib import Path
import pickle

import torch

from training import train_model
from model import PeptideCVAE
from inference import generate_peptides

In [ ]:
# Paths relative to this notebook
baseline_dir = Path(".")
dataset_file = "../database/test.json"
model_path = baseline_dir / "cvae_peptide_model.pth"
scaler_path = baseline_dir / "scaler.pkl"

# Quick 1-epoch train
model, device = train_model(
    dataset_file=str(dataset_file),
    epochs=50,
    batch_size=32,
    max_len=12,
    latent_dim=32,
    model_path=str(model_path),
)

print(f"Model saved at: {model_path}")
print(f"Scaler expected at: {scaler_path}")

In [ ]:
# Reload model + scaler and generate a few peptides
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

# Keep architecture args aligned with training defaults
loaded_model = PeptideCVAE(
    vocab_size=24,
    condition_dim=8,
    max_seq_len=14,
    latent_dim=32,
)
loaded_model.load_state_dict(torch.load(model_path, map_location=device))

# Trying to imitate this:
# {
#     "dbaasp_id": "DBAASPS_373",
#     "sequence": "KLFKRWKHLFR",
#     "length": 11,
#     "smiles": "CC(C)C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N)CCCCN)C(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](CCCN=C(N)N)C(=O)O",
#     "ph": null,
#     "molecular_weight": 1558.9480000000003,
#     "logp": -0.992100000000006,
#     "net_charge": 5.0,
#     "isoelectric_point": 12.18,
#     "hydrophobicity": 1.05,
#     "cathionicity": 6,
#     "target_groups": [
#         "Gram+"
#     ],
#     "complexity": "Monomer"
# }
target = [11, 7, 1558.94, -0.9921, 5.0, 12.18, 1.05, 6]

samples = generate_peptides(
    model=loaded_model,
    scaler=scaler,
    num_samples=5,
    properties=target,
    temperature=0.9,
    top_k=5,
)

samples